In [1]:
import os
from gensim.models import KeyedVectors

# --- ファイルパスの設定 ---
# モデルと評価ファイルがコードと同じディレクトリにあることを想定
MODEL_FILE = "GoogleNews-vectors-negative300.bin.gz"
QUESTIONS_FILE = "questions-words.txt"

# --- ファイルの存在チェック ---
if not os.path.exists(MODEL_FILE):
    print(f"エラー: モデルファイルが見つかりません: {MODEL_FILE}")
    print("Google Newsの事前学習済みモデルをダウンロードして、同じディレクトリに配置してください。")
elif not os.path.exists(QUESTIONS_FILE):
    print(f"エラー: 評価ファイルが見つかりません: {QUESTIONS_FILE}")
    print("questions-words.txt をダウンロードして、同じディレクトリに配置してください。")
else:
    try:
        # --- モデルの読み込み ---
        print("モデルを読み込んでいます... (数分かかる場合があります)")
        model = KeyedVectors.load_word2vec_format(MODEL_FILE, binary=True)
        print("モデルの読み込みが完了しました。")

        # --- アナロジー評価の実行 ---
        # 結果を格納する辞書
        results = {
            'semantic': {'correct': 0, 'total': 0},
            'syntactic': {'correct': 0, 'total': 0}
        }

        # カテゴリの分類
        # 'gram'で始まるカテゴリを文法的(syntactic)、それ以外を意味的(semantic)とする
        current_category_type = None

        with open(QUESTIONS_FILE, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue

                # カテゴリ行の処理
                if line.startswith(":"):
                    category_name = line.split()[1]
                    if category_name.startswith("gram"):
                        current_category_type = 'syntactic'
                    else:
                        current_category_type = 'semantic'
                    print(f"\nProcessing {current_category_type} section: {category_name}")
                    continue

                # アナロジー問題の処理
                if current_category_type is None:
                    continue

                words = line.split()
                if len(words) != 4:
                    continue

                a, b, c, expected_d = words

                # 総問題数をカウント
                results[current_category_type]['total'] += 1

                try:
                    # most_similarで最も類似する単語を1つ取得
                    # b - a + c のアナロジー
                    predicted_result = model.most_similar(positive=[b, c], negative=[a], topn=1)
                    predicted_d = predicted_result[0][0]

                    # 正解かどうかを判定（小文字に統一して比較）
                    if predicted_d.lower() == expected_d.lower():
                        results[current_category_type]['correct'] += 1

                except KeyError:
                    # モデルの語彙にない単語が含まれていた場合はスキップ
                    pass

        # --- 結果の表示 ---
        print("\n--- Analogy Task Results ---")

        # 意味的アナロジーの正解率
        sem_total = results['semantic']['total']
        sem_correct = results['semantic']['correct']
        if sem_total > 0:
            sem_accuracy = (sem_correct / sem_total) * 100
            print(f"Semantic analogy accuracy: {sem_accuracy:.2f}% ({sem_correct} / {sem_total})")
        else:
            print("Semantic analogy: No questions found.")

        # 文法的アナロジーの正解率
        syn_total = results['syntactic']['total']
        syn_correct = results['syntactic']['correct']
        if syn_total > 0:
            syn_accuracy = (syn_correct / syn_total) * 100
            print(f"Syntactic analogy accuracy: {syn_accuracy:.2f}% ({syn_correct} / {syn_total})")
        else:
            print("Syntactic analogy: No questions found.")

        # 総合正解率
        total_questions = sem_total + syn_total
        total_correct = sem_correct + syn_correct
        if total_questions > 0:
            overall_accuracy = (total_correct / total_questions) * 100
            print(f"Overall accuracy: {overall_accuracy:.2f}% ({total_correct} / {total_questions})")

    except Exception as e:
        print(f"処理中に予期せぬエラーが発生しました: {e}")

モデルを読み込んでいます... (数分かかる場合があります)
モデルの読み込みが完了しました。

Processing semantic section: capital-common-countries

Processing semantic section: capital-world

Processing semantic section: currency

Processing semantic section: city-in-state

Processing semantic section: family

Processing syntactic section: gram1-adjective-to-adverb

Processing syntactic section: gram2-opposite

Processing syntactic section: gram3-comparative

Processing syntactic section: gram4-superlative

Processing syntactic section: gram5-present-participle

Processing syntactic section: gram6-nationality-adjective

Processing syntactic section: gram7-past-tense

Processing syntactic section: gram8-plural

Processing syntactic section: gram9-plural-verbs

--- Analogy Task Results ---
Semantic analogy accuracy: 73.09% (6482 / 8869)
Syntactic analogy accuracy: 74.01% (7901 / 10675)
Overall accuracy: 73.59% (14383 / 19544)
